In [ ]:
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

import chromadb

import gradio as gr

In [1]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY was not found. Check your .env file.")

print("OpenAI API key loaded successfully.")

NameError: name 'load_dotenv' is not defined

In [2]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("OpenAI embeddings initialized successfully.")

NameError: name 'OpenAIEmbeddings' is not defined

In [3]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

print("GPT-3.5 Turbo initialized successfully.")

NameError: name 'ChatOpenAI' is not defined

In [4]:
pdf_path = "the_nestle_hr_policy_pdf_2012.pdf"

print(f"PDF path: {pdf_path}")

PDF path: the_nestle_hr_policy_pdf_2012.pdf


In [5]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")

NameError: name 'PyPDFLoader' is not defined

In [ ]:
print(documents[0].page_content)

In [ ]:
print(documents[0].metadata)

In [ ]:
for i, doc in enumerate(documents):
    print(f"\n--- Page {i + 1} ---")
    print(doc.page_content[:500])

In [ ]:
total_characters = sum(len(doc.page_content) for doc in documents)

print(f"Pages loaded: {len(documents)}")
print(f"Total characters extracted: {total_characters}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter created successfully.")

In [ ]:
chunks = text_splitter.split_documents(documents)

print(f"Original pages: {len(documents)}")
print(f"Total chunks created: {len(chunks)}")

In [ ]:
print(chunks[0].page_content)

In [ ]:
print(chunks[0].metadata)

In [ ]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)
    print(f"Metadata: {chunk.metadata}")

In [ ]:
chunk_sizes = [len(chunk.page_content) for chunk in chunks]

print(f"Number of chunks: {len(chunk_sizes)}")
print(f"Smallest chunk: {min(chunk_sizes)} characters")
print(f"Largest chunk: {max(chunk_sizes)} characters")
print(f"Average chunk: {sum(chunk_sizes) / len(chunk_sizes):.2f} characters")

In [6]:
from langchain_chroma import Chroma

print("Chroma imported successfully.")

Chroma imported successfully.


In [8]:
print("Embedding model initialized:", embeddings.model)

NameError: name 'embeddings' is not defined

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nestle_hr_policy"
)

print("ChromaDB vector store created successfully.")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")

In [ ]:
query = "What does Nestlé consider when hiring employees?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved documents: {len(retrieved_docs)}")

In [ ]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(doc.page_content)
    print("\nMetadata:")
    print(doc.metadata)

In [ ]:
query = "What does Nestlé say about employee training and development?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(doc.page_content)

In [ ]:
from langchain_core.prompts import PromptTemplate

print("PromptTemplate imported successfully.")

In [ ]:
prompt_template = """
You are an AI-powered HR assistant for Nestlé.

Answer the user's question using only the information provided
in the Nestlé Human Resources Policy.

If the answer cannot be found in the provided context, clearly
state that the information is not available in the provided
policy document.

Do not make up or assume information.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt template created successfully.")

In [ ]:
def format_documents(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

print("Document formatting function created successfully.")

In [ ]:
def answer_question(question):
    retrieved_docs = retriever.invoke(question)

    context = format_documents(retrieved_docs)

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(final_prompt)

    return response.content

In [ ]:
question = "What does Nestlé consider when hiring an employee?"

answer = answer_question(question)

print(answer)

In [ ]:
question = "How does Nestlé approach employee training and development?"

answer = answer_question(question)

print(answer)

In [ ]:
question = "What is Nestlé's employee parking policy?"

answer = answer_question(question)

print(answer)

In [ ]:
gradio_app = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(
        label="Ask a Nestlé HR Policy Question",
        placeholder="Enter your HR policy question here..."
    ),
    outputs=gr.Textbox(
        label="HR Assistant Answer"
    ),
    title="Nestlé HR Policy Assistant",
    description=(
        "Ask questions about the Nestlé Human Resources Policy. "
        "Answers are generated using information retrieved from "
        "the provided HR policy document."
    )
)

print("Gradio interface created successfully.")

In [ ]:
gradio_app.launch()